# KisanLens – model training (Google Colab)

Trains the tomato/potato leaf classifier with MobileNetV2 transfer learning, evaluates it on a held-out test split
(group-aware, so augmented copies of one photo never straddle splits), tunes uncertainty thresholds on the validation
split, and exports `saved_model.keras`, `class_names.json`, `thresholds.json` and the measured metrics.

**Before you start:** Runtime → Change runtime type → GPU. Read `data/README.md` about the PlantVillage license.

## 1. Get the project code
Upload the `kisanlens` folder (zipped) to Colab, or clone your Git repo.

In [ ]:
# Option A: upload kisanlens.zip using the Files panel, then:
!unzip -q -o kisanlens.zip -d /content
%cd /content/kisanlens
# Option B: !git clone <your-repo-url> /content/kisanlens && %cd /content/kisanlens

In [ ]:
!pip install -q -r requirements.txt

## 2. Get the data
Pick ONE option. The folder must end up as `data/PlantVillage/<class folders>` (see `data/README.md`).

In [ ]:
# Option A – from Google Drive (upload the PlantVillage folder there first)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data && ln -sfn "/content/drive/MyDrive/PlantVillage" data/PlantVillage

In [ ]:
# Option B – Kaggle (needs your own kaggle.json API token uploaded to /content)
# !pip install -q kaggle
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d <owner/plantvillage-dataset-slug> -p data --unzip

In [ ]:
!ls data/PlantVillage | head -50

## 3. Train + evaluate
Typical run: about 20–40 minutes on a Colab GPU. Everything reported below is measured, not hard-coded.

In [ ]:
!python -m model.train_model --data-dir data/PlantVillage --epochs 8 --fine-tune-epochs 6

## 4. Inspect the measured results

In [ ]:
import json
m = json.load(open('model/artifacts/metrics.json'))
print('Test accuracy :', round(m['test_accuracy'], 4))
print('Test macro F1 :', round(m['test_macro_f1'], 4))
print('Thresholds    :', m['thresholds'])
print('On validation :', m['thresholds_on_validation'])
print('On test       :', m['thresholds_on_test'])
print(open('model/artifacts/classification_report.txt').read())

In [ ]:
from IPython.display import Image
Image('model/artifacts/confusion_matrix.png')

## 5. Test on REAL field photos (do this before calling it field-ready)
Upload your own labelled photos to `data/realworld/<class_key>/` (see `data/README.md`).

In [ ]:
!python -m model.evaluate_realworld --dir data/realworld --out model/artifacts/realworld_results.json

## 6. Download the files for the web app

In [ ]:
!cd model && zip -r ../kisanlens_model.zip saved_model.keras class_names.json thresholds.json artifacts -x 'artifacts/splits.csv'
from google.colab import files
files.download('kisanlens_model.zip')

Unzip the download into the project's `model/` folder (so `model/saved_model.keras` exists), then run `python app.py`.

**Honest reporting:** quote the numbers above together with their limits. Test accuracy on PlantVillage-style photos is
not evidence of field accuracy. Report the real-world result separately.